### Загрузка данных

In [8]:
import json
from pathlib import Path
import pandas as pd

path = Path("./data/df_text2graph.jsonl")


records = [json.loads(line) for line in path.open("r", encoding="utf-8")]
df = pd.DataFrame(records)

display(df.head())
print(df.dtypes)

,id,input,output
0,text_00001,"Проект закрыли, бюджет иссяк. Команду перевели...",VERTEX_LIST:\nV1: закрыли Проект\nV2: перевели...
1,text_00002,"новым алгоритмам, точность модели выросла. Оши...",VERTEX_LIST:\nV1: точность выросла\nV2: Ошибки...
2,text_00003,"Дорогу перекрыли, шёл капитальный ремонт моста...",VERTEX_LIST:\nV1: перекрыли Дорогу\nV2: органи...
3,text_00004,"Мы отложили встречу, ключевой участник внезапн...",VERTEX_LIST:\nV1: Мы отложили встречу\nV2: уча...
4,text_00005,"погода резко испортилась, вылет отменили. Пасс...",VERTEX_LIST:\nV1: погода испортилась резко\nV2...


id        object
input     object
output    object
dtype: object


In [ ]:
import re

def parse_output(text):
    vertices = {}
    edges = []

    if not text:
        return vertices, edges

    parts = text.split("RELATIONSHIPS_LIST:")
    vblock = parts[0].replace("VERTEX_LIST:", "").strip()
    rblock = parts[1].strip() if len(parts) > 1 else ""

    for line in vblock.splitlines():
        line = line.strip()
        if not line:
            continue

        m = re.match(r"(V\d+)\s*:\s*(.+)", line)
        if m:
            vertices[m.group(1)] = m.group(2).strip()

    for line in rblock.splitlines():
        line = line.strip()
        if not line or "тип связи не определён" in line.lower():
            continue
        m = re.match(r"(V\d+)\s*->\s*(V\d+)\s+(\w+)", line)
        if m:
            edges.append((m.group(1), m.group(2), m.group(3)))

    return vertices, edges

example = {
    "id": "text_00001",
    "input": "Проект закрыли, бюджет иссяк. Команду перевели...",
    "output": "VERTEX_LIST:\nV1: закрыли Проект\nV2: перевели Команду\n\nRELATIONSHIPS_LIST:\nV2->V1 CAUSES"
}

v, e = parse_output(example["output"])
print(v)
print(e)


{'V1': 'закрыли Проект', 'V2': 'перевели Команду'}
[('V2', 'V1', 'CAUSES')]


### Обучение модели

In [4]:
from datasets import Dataset

def make_seq2seq_dataset(df):
    return Dataset.from_dict({
        "id": df["id"].tolist(),
        "input_text": df["input"].tolist(),
        "target_text": df["output"].tolist()
    })

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

model_name = "cointegrated/rut5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def preprocess(example):
    model_inputs = tokenizer(example["input_text"], max_length=256, truncation=True)
    labels = tokenizer(example["target_text"], max_length=256, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset = make_seq2seq_dataset(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

tokenized = dataset.map(preprocess, batched=False)

args = TrainingArguments(
    output_dir="./rut5",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    num_train_epochs=6,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    logging_steps=50,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
)

trainer.train()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/8837 [00:00<?, ? examples/s]

Map:   0%|          | 0/982 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


In [ ]:
import shutil

save_dir = "./rut5_saved"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Сохранено в", save_dir)


shutil.make_archive("rut5_saved", "zip", save_dir)
print("Готово: rut5_saved.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Сохранено в ./rut5_saved


### Тестирование модели

In [10]:
import random
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

def show_examples(model, tokenizer, df, n=5, max_len=256, seed=42):
    random.seed(seed)
    idxs = random.sample(range(len(df)), k=min(n, len(df)))

    device = next(model.parameters()).device
    model.eval()

    for i in idxs:
        row = df.iloc[i]
        inp = row["input"]
        gold = row["output"]

        inputs = tokenizer(inp, return_tensors="pt", truncation=True, max_length=max_len)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            gen_ids = model.generate(**inputs, max_length=max_len)

        pred = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

        print()
        print("INPUT:")
        print(inp)
        print("\nEXPECTED:")
        print(gold)
        print("\nPREDICTED:")
        print(pred)
        print()


MODEL_PATH = "./rut5_saved"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH, local_files_only=True)
model.eval()


show_examples(model, tokenizer, df, n=5)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



INPUT:
Вчера водитель объехал пробку, в прошлом году навигатор предложил альтернативу. Неделю назад пассажиры сэкономили время, сегодня маршрут оказался короче.

EXPECTED:
VERTEX_LIST:
V1: Вчера водитель объехал пробку Вчера
V2: навигатор предложил альтернативу
V3: Неделю пассажиры сэкономили время
V4: сегодня маршрут оказался сегодня

RELATIONSHIPS_LIST:
V2->V1 CAUSES
V4->V3 CAUSES

PREDICTED:
VERTEX_LIST: V1: вчера водитель объехал пробку Вчера V2: навигатор предложил альтернативу V3: Неделю пассажиры сэкономили время V4: сегодня маршрут оказался сегодня RELATIONSHIPS_LIST: V2->V1 CAUSES V4->V3 CAUSES


INPUT:
Водители объезжали пробку, авария перекрыла полосу. Полиция оформила документы, участники не соглашались.

EXPECTED:
VERTEX_LIST:
V1: Водители объезжали пробку
V2: авария перекрыла полосу
V3: Полиция оформила документы

RELATIONSHIPS_LIST:
V2->V1 CAUSES

PREDICTED:
VERTEX_LIST: V1: Водители объезжали пробку V2: авария перекрила полосу V3: Полиция оформила документы RELATIONSHI